In [ ]:
# Install required libraries
!pip install -U langchain langchain-community langchain-groq langchain-text-splitters langchain-chroma langchain-classic flashrank pypdf duckduckgo-search ddgs

In [ ]:
#import packages
from langchain_groq import ChatGroq
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from google.colab import userdata
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_community.document_compressors.flashrank_rerank import FlashrankRerank
from langchain_classic.chains import ConversationalRetrievalChain
from langchain_classic.memory import ConversationBufferWindowMemory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_classic.agents import create_tool_calling_agent, AgentExecutor
from langchain_core.tools import create_retriever_tool
import re
from datetime import datetime


In [ ]:
file=f"/content/diagnostic-tools.pdf"
loader = PyPDFLoader(file)
pages = loader.load()

In [ ]:
print(pages)
print(type(pages))
print(pages[0])
print(len(pages))
print(pages[0].metadata)

In [ ]:
#splitting texts for vectorization
GROQ_API_KEY = userdata.get('GROQ_API_KEY') # Make sure your secret is named 'GROQ_API_KEY'
llm = ChatGroq(api_key=GROQ_API_KEY, model="llama-3.1-8b-instant") # to frame answer based on conetxt and questions
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

text_splitter=RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
chunks=text_splitter.split_documents(pages)

In [ ]:
#create Chroma collection
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore=Chroma.from_documents(documents=chunks,embedding=embeddings,collection_name="aws_case_investigator")

In [ ]:
#validate basic outputs from vectorstore
response=vectorstore.similarity_search_with_score("what is this docment about")
print(response)

In [ ]:
SYSTEM_PROMPT = """
You are an AI Customer Support Case Investigator.

You are given multiple past incident cases.

Your job is NOT just to answer, but to:

1. Identify common patterns across cases
2. Detect if issue is recurring
3. Infer most likely root cause
4. Suggest best resolution

If multiple cases are similar, highlight the pattern.

If no strong match, say "Insufficient data".

Return structured output:

Root Cause:
Resolution:
Recurring Pattern:
Confidence (High/Medium/Low):

Context:
{context}
"""

prompt = ChatPromptTemplate.from_messages([("system", SYSTEM_PROMPT),("placeholder", "{chat_history}"),("human", "{question}")])
memory = ConversationBufferWindowMemory(memory_key="chat_history",k=5,return_messages=True,output_key="answer")
chain = ConversationalRetrievalChain.from_llm(llm=llm,retriever=vectorstore.as_retriever(search_kwargs={"k": 5}),memory=memory,return_source_documents=True,combine_docs_chain_kwargs={"prompt": prompt})

# Test conversation
def chat(question):
    result = chain.invoke({"question": question})
    print(f"Q: {question}")
    print(f"A: {result['answer']}")
    print(f"Sources: {len(result['source_documents'])} chunks")
    print(f"Memory has: {len(memory.chat_memory.messages)} messages")
    print("---")

In [ ]:
#test questions
chat("what is db high cpu utilization issue?")

In [ ]:
chat("is there any new feature added in Aurora postgres version in latest release to improve latency?")

Below cells extend the RAG pipeline with validation, time/version-aware checks,
external enrichment, and agent-driven decision logic for response refinement.

In [ ]:
#This cell has function used to process time based, question,revalidation and merge function to used with responses and use of agants calling
OUTDATED_SIGNALS = [
    "latest", "new feature", "recent", "recently", "current version",
    "latest release", "new release", "changelog", "update", "upgrade",
    "what's new", "introduced in", "added in", "announced", "2024", "2025",
    "roadmap", "upcoming", "just released", "new in", "enhancement",
    "improvement", "patch", "version"
]

def is_question_time_sensitive(question: str) -> bool:

    question_lower = question.lower()
    matched = [kw for kw in OUTDATED_SIGNALS if kw in question_lower]
    if matched:
        print(f"[Outdated Check] Time-sensitive signals found: {matched}")
    return len(matched) > 0


def is_answer_uncertain(answer: str) -> bool:

    uncertainty_phrases = [
        "i don't have", "insufficient data", "not mentioned",
        "no information", "cannot find", "not in the context",
        "i'm not sure", "unclear", "not available in"
    ]
    answer_lower = answer.lower()
    matched = [p for p in uncertainty_phrases if p in answer_lower]
    if matched:
        print(f"[Outdated Check] Answer uncertainty signals found: {matched}")
    return len(matched) > 0



In [ ]:
def chat(question: str):
    """
    Enhanced chat function:
    1. Runs the RAG chain against ChromaDB (internal docs)
    2. Passes answer through outdated validator
    3. If time-sensitive, agent enriches with live web data
    4. Returns merged, validated final answer
    """
    print(f"\n{'='*60}")
    print(f"Q: {question}")
    print(f"{'='*60}")

    # Step 1: Get RAG answer from internal docs
    print("[Step 1] Querying internal ChromaDB (RAG)...")
    rag_result = chain.invoke({"question": question})
    rag_answer = rag_result["answer"]
    source_count = len(rag_result["source_documents"])

    print(f"[Step 1] RAG retrieved {source_count} source chunks")

    # Step 2: Validate freshness and enrich if needed
    print("[Step 2] Running outdated response check...")
    final_answer = validate_and_enrich(question, rag_answer)

    print(f"\nFINAL ANSWER:\n{final_answer}")
    print(f"\nMemory: {len(memory.chat_memory.messages)} messages in window")
    print("=" * 60)
    return final_answer

In [ ]:

def validate_and_enrich(question: str, rag_answer: str) -> str:
    """
    Takes the RAG chain answer and checks if it needs freshness validation.

    Logic:
    1. If question has time-sensitive keywords  → trigger web agent
    2. If RAG answer signals uncertainty        → trigger web agent
    3. Otherwise                                → return RAG answer as-is

    When triggered, the agent searches web + internal docs,
    then the final answer merges both perspectives.
    """

    needs_validation = (
        is_question_time_sensitive(question) or
        is_answer_uncertain(rag_answer)
    )

    if not needs_validation:
        print("[Validator] RAG answer looks current. No web validation needed.")
        return rag_answer

    print("[Validator] Triggering web validation agent for fresh data...")
    print("-" * 50)


In [ ]:
web_search_tool = DuckDuckGoSearchRun()
web_search_tool.description = (
    "Use this to find recent, up-to-date information about cloud services, "
    "AWS features, Aurora PostgreSQL releases, and any technical enhancements "
    "that may not be in the internal knowledge base."
)

retriever_tool = create_retriever_tool(
    vectorstore.as_retriever(search_kwargs={"k": 5}),
    name="search_internal_cases",
    description=(
        "Search internal case history and diagnostic documents. "
        "Use this for known issues, past incidents, root cause patterns, "
        "and internal resolution steps."
    )
)

agent_tools = [retriever_tool, web_search_tool]

# Agent prompt — instructs it to combine internal + web knowledge
AGENT_SYSTEM_PROMPT = """
You are an AI Cloud Support Assistant with access to two tools:

1. search_internal_cases — searches internal incident history and documentation
2. DuckDuckGoSearch — searches the live web for recent updates and new features

When the question involves recent releases, new features, or version-specific changes:
- ALWAYS search the web to verify the latest information
- ALSO check internal cases for related historical context
- Combine both into one clear, structured answer

Always return:
Root Cause / Feature Summary:
Resolution / What Changed:
Source: [Internal Docs / Web / Both]
Confidence: [High / Medium / Low]
As of: {current_date}
""".format(current_date=datetime.now().strftime("%B %Y"))

agent_prompt = ChatPromptTemplate.from_messages([
    ("system", AGENT_SYSTEM_PROMPT),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])

# Separate memory for agent (expects 'output' key, not 'answer')
agent_memory = ConversationBufferWindowMemory(
    memory_key="chat_history",
    k=5,
    return_messages=True,
    output_key="output"
)

agent = create_tool_calling_agent(llm, agent_tools, agent_prompt)
agent_executor = AgentExecutor(
    agent=agent,
    tools=agent_tools,
    memory=agent_memory,
    verbose=True,           # set False to hide tool call details
    handle_parsing_errors=True
)

def agent_chat(question: str, rag_answer: str):
    # Build a focused query for the agent
    agent_query = (
        f"The following question may require up-to-date information. "
        f"Search for recent enhancements, releases, or changes relevant to it.\n\n"
        f"Question: {question}\n\n"
        f"Internal RAG answer (may be outdated):\n{rag_answer}\n\n"
        f"Please validate and enrich the above with the latest available information."
    )

    agent_result = agent_executor.invoke({"input": agent_query})
    web_enriched_answer = agent_result["output"]

    # Merge RAG answer + web-enriched answer into one response
    final_answer = f"""
INTERNAL KNOWLEDGE BASE (RAG)
{rag_answer}

VALIDATED WITH LIVE WEB (Agent)
{web_enriched_answer}

VALIDATION STATUS
Checked on : {datetime.now().strftime("%d %B %Y, %H:%M")}
Trigger     : Time-sensitive question detected
Action      : Web search agent invoked for freshness check
"""
    return final_answer

In [ ]:

def chat(question: str):
    """
    Enhanced chat function:
    1. Runs the RAG chain against ChromaDB (internal docs)
    2. Passes answer through outdated validator
    3. If time-sensitive, agent enriches with live web data
    4. Returns merged, validated final answer
    """
    print(f"\n{'='*60}")
    print(f"Q: {question}")
    print(f"{'='*60}")

    # Step 1: Get RAG answer from internal docs
    print("[Step 1] Querying internal ChromaDB (RAG)...")
    rag_result = chain.invoke({"question": question})
    rag_answer = rag_result["answer"]
    source_count = len(rag_result["source_documents"])

    print(f"[Step 1] RAG retrieved {source_count} source chunks")

    # Step 2: Validate freshness and enrich if needed
    print("[Step 2] Running outdated response check...")
    final_answer = validate_and_enrich(question, rag_answer)

    print(f"\nFINAL ANSWER:\n{final_answer}")
    print(f"\nMemory: {len(memory.chat_memory.messages)} messages in window")
    print("=" * 60)
    return final_answer


In [ ]:

# This should NOT trigger the web agent (no time-sensitive keywords)
chat("what is db high cpu utilization issue?")

# This SHOULD trigger the web agent (contains "new feature", "latest release")
chat("is there any new feature added in Aurora postgres version in latest release to improve latency?")

# Another time-sensitive trigger
chat("what enhancements were recently announced for Aurora PostgreSQL 16?")

# Run this on any question BEFORE chatting to preview the decision

def preview_validation_decision(question: str):
    """Quick check — will this question trigger the web agent or not?"""
    time_sensitive = is_question_time_sensitive(question)
    print(f"Question     : {question}")
    print(f"Time-sensitive: {'YES → web agent will be called' if time_sensitive else 'NO → RAG only'}")

preview_validation_decision("what is the root cause of memory pressure errors?")
preview_validation_decision("what new features were added in Aurora PostgreSQL 15.4?")
preview_validation_decision("is there a recent patch for RDS connection pooling?")